# Daily Activity Metrics

Bu notebook, full clean `stg_wowah_events` part dosyalarindan gunluk oyuncu aktivite metriklerini uretir.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

print('DuckDB ve pandas hazir.')


In [ ]:
project_root = Path.cwd()
processed_dir = project_root / 'data/processed'
outputs_dir = project_root / 'data/outputs'
sql_file = project_root / 'sql/02_fct_player_daily_and_daily_metrics.sql'

clean_parts_dir = processed_dir / 'stg_wowah_events_clean_parts'
clean_part_files = sorted(clean_parts_dir.glob('*.parquet'))
if not clean_part_files:
    raise FileNotFoundError('Full clean staging part dosyalari bulunamadi.')

stg_source_glob = str(clean_parts_dir / '*.parquet')
preview_avoided = 'preview_stg_wowah_events.parquet' not in stg_source_glob

fct_output_path = processed_dir / 'fct_player_daily.parquet'
agg_output_path = processed_dir / 'agg_daily_metrics.parquet'
validation_output_path = outputs_dir / 'daily_metrics_validation_summary.csv'

print('Source glob:', stg_source_glob)
print('Preview avoided:', 'yes' if preview_avoided else 'no')
print('Part file count:', len(clean_part_files))
print('SQL file:', sql_file)


In [ ]:
con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE VIEW stg_wowah_events AS SELECT * FROM read_parquet('{stg_source_glob}')"
)

source_validation_overview = con.execute(
    '''
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT avatar_id) AS unique_avatar_id,
        MIN(observed_at) AS min_observed_at,
        MAX(observed_at) AS max_observed_at,
        COUNT(DISTINCT CAST(observed_at AS DATE)) AS distinct_activity_dates,
        COUNT(*) - COUNT(DISTINCT (CAST(observed_at AS VARCHAR) || '|' || CAST(avatar_id AS VARCHAR))) AS duplicate_observed_at_avatar_id_count,
        AVG(CASE WHEN observed_at IS NULL THEN 1.0 ELSE 0.0 END) AS observed_at_null_rate,
        AVG(CASE WHEN avatar_id IS NULL THEN 1.0 ELSE 0.0 END) AS avatar_id_null_rate,
        AVG(CASE WHEN level IS NULL THEN 1.0 ELSE 0.0 END) AS level_null_rate,
        AVG(CASE WHEN race IS NULL THEN 1.0 ELSE 0.0 END) AS race_null_rate,
        AVG(CASE WHEN character_class IS NULL THEN 1.0 ELSE 0.0 END) AS character_class_null_rate,
        AVG(CASE WHEN zone IS NULL THEN 1.0 ELSE 0.0 END) AS zone_null_rate,
        MIN(level) AS level_min,
        MAX(level) AS level_max
    FROM stg_wowah_events
    '''
).df()

top_race_values = con.execute(
    '''
    SELECT race, COUNT(*) AS row_count
    FROM stg_wowah_events
    GROUP BY 1
    ORDER BY 2 DESC, 1 ASC
    LIMIT 10
    '''
).df()

top_character_class_values = con.execute(
    '''
    SELECT character_class, COUNT(*) AS row_count
    FROM stg_wowah_events
    GROUP BY 1
    ORDER BY 2 DESC, 1 ASC
    LIMIT 10
    '''
).df()

top_zone_values = con.execute(
    '''
    SELECT zone, COUNT(*) AS row_count
    FROM stg_wowah_events
    GROUP BY 1
    ORDER BY 2 DESC, 1 ASC
    LIMIT 10
    '''
).df()

source_validation_overview


In [ ]:
print('Top 10 race values')
print(top_race_values)
print('\nTop 10 character_class values')
print(top_character_class_values)
print('\nTop 10 zone values')
print(top_zone_values)


In [ ]:
distinct_activity_dates = int(source_validation_overview.loc[0, 'distinct_activity_dates'])
if distinct_activity_dates <= 2:
    raise ValueError('Source still looks like preview data: distinct activity dates <= 2.')

print('Sanity check gecti. Distinct activity dates:', distinct_activity_dates)


## Build analytics views

SQL dosyasi okunur ve `fct_player_daily` ile `agg_daily_metrics` DuckDB view'leri olusturulur.

In [ ]:
analytics_sql = sql_file.read_text()
con.execute(analytics_sql)

view_counts = con.execute(
    '''
    SELECT 'fct_player_daily' AS view_name, COUNT(*) AS row_count FROM fct_player_daily
    UNION ALL
    SELECT 'agg_daily_metrics' AS view_name, COUNT(*) AS row_count FROM agg_daily_metrics
    '''
).df()

view_counts


In [ ]:
con.execute(
    '''
    SELECT *
    FROM fct_player_daily
    ORDER BY activity_date, avatar_id
    LIMIT 20
    '''
).df()


In [ ]:
con.execute(
    '''
    SELECT *
    FROM agg_daily_metrics
    ORDER BY activity_date
    LIMIT 20
    '''
).df()


In [ ]:
validation_overview = con.execute(
    '''
    WITH fct_stats AS (
        SELECT
            COUNT(*) AS fct_player_daily_rows,
            COUNT(DISTINCT avatar_id) AS distinct_avatars,
            MIN(activity_date) AS min_activity_date,
            MAX(activity_date) AS max_activity_date,
            SUM(CASE WHEN level_gain_day < 0 THEN 1 ELSE 0 END) AS negative_level_gain_days
        FROM fct_player_daily
    ),
    agg_stats AS (
        SELECT
            MIN(dau) AS dau_min,
            MAX(dau) AS dau_max,
            AVG(dau * 1.0) AS dau_avg,
            MIN(mau) AS mau_min,
            MAX(mau) AS mau_max,
            AVG(mau * 1.0) AS mau_avg,
            SUM(CASE WHEN mau = 0 THEN 1 ELSE 0 END) AS days_with_zero_mau,
            SUM(CASE WHEN dau > mau THEN 1 ELSE 0 END) AS days_where_dau_gt_mau,
            COUNT(*) AS total_days
        FROM agg_daily_metrics
    )
    SELECT *
    FROM fct_stats, agg_stats
    '''
).df()

validation_overview


In [ ]:
top_10_days_by_dau = con.execute(
    '''
    SELECT *
    FROM agg_daily_metrics
    ORDER BY dau DESC, activity_date ASC
    LIMIT 10
    '''
).df()

top_10_days_by_dau


In [ ]:
fct_output_tmp = fct_output_path.with_suffix(fct_output_path.suffix + '.tmp')
agg_output_tmp = agg_output_path.with_suffix(agg_output_path.suffix + '.tmp')
validation_output_tmp = validation_output_path.with_suffix(validation_output_path.suffix + '.tmp')

for temp_path in [fct_output_tmp, agg_output_tmp, validation_output_tmp]:
    if temp_path.exists():
        temp_path.unlink()

con.execute(
    f"COPY (SELECT * FROM fct_player_daily ORDER BY activity_date, avatar_id) TO '{fct_output_tmp}' (FORMAT PARQUET)"
)
con.execute(
    f"COPY (SELECT * FROM agg_daily_metrics ORDER BY activity_date) TO '{agg_output_tmp}' (FORMAT PARQUET)"
)

con.execute(
    '''
    CREATE OR REPLACE VIEW daily_metrics_validation_summary AS
    WITH summary_rows AS (
        SELECT
            'summary' AS section,
            1 AS position,
            'fct_player_daily_row_count' AS metric_name,
            CAST(COUNT(*) AS VARCHAR) AS metric_value,
            NULL::DATE AS activity_date,
            NULL::BIGINT AS dau,
            NULL::BIGINT AS wau,
            NULL::BIGINT AS mau,
            NULL::DOUBLE AS dau_mau_stickiness,
            NULL::BIGINT AS total_observations,
            NULL::BIGINT AS active_guild_members,
            NULL::DOUBLE AS guild_member_share,
            NULL::DOUBLE AS avg_observations_per_avatar,
            NULL::DOUBLE AS avg_level,
            NULL::DOUBLE AS avg_level_gain_day,
            NULL::BIGINT AS total_level_gain_day,
            NULL::BIGINT AS new_avatars,
            NULL::BIGINT AS returning_avatars,
            NULL::BIGINT AS reactivated_avatars_30d
        FROM fct_player_daily
        UNION ALL
        SELECT 'summary', 2, 'distinct_avatars', CAST(COUNT(DISTINCT avatar_id) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM fct_player_daily
        UNION ALL
        SELECT 'summary', 3, 'min_activity_date', CAST(MIN(activity_date) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM fct_player_daily
        UNION ALL
        SELECT 'summary', 4, 'max_activity_date', CAST(MAX(activity_date) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM fct_player_daily
        UNION ALL
        SELECT 'summary', 5, 'dau_min', CAST(MIN(dau) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 6, 'dau_max', CAST(MAX(dau) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 7, 'dau_avg', CAST(AVG(dau * 1.0) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 8, 'mau_min', CAST(MIN(mau) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 9, 'mau_max', CAST(MAX(mau) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 10, 'mau_avg', CAST(AVG(mau * 1.0) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 11, 'days_where_mau_is_zero', CAST(SUM(CASE WHEN mau = 0 THEN 1 ELSE 0 END) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 12, 'days_where_dau_gt_mau', CAST(SUM(CASE WHEN dau > mau THEN 1 ELSE 0 END) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM agg_daily_metrics
        UNION ALL
        SELECT 'summary', 13, 'negative_level_gain_day_count', CAST(SUM(CASE WHEN level_gain_day < 0 THEN 1 ELSE 0 END) AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM fct_player_daily
    ),
    top_days AS (
        SELECT
            'top_10_days_by_dau' AS section,
            ROW_NUMBER() OVER (ORDER BY dau DESC, activity_date ASC) AS position,
            'top_day' AS metric_name,
            CAST(dau AS VARCHAR) AS metric_value,
            activity_date,
            dau,
            wau,
            mau,
            dau_mau_stickiness,
            total_observations,
            active_guild_members,
            guild_member_share,
            avg_observations_per_avatar,
            avg_level,
            avg_level_gain_day,
            total_level_gain_day,
            new_avatars,
            returning_avatars,
            reactivated_avatars_30d
        FROM agg_daily_metrics
        ORDER BY dau DESC, activity_date ASC
        LIMIT 10
    ),
    first_twenty AS (
        SELECT
            'agg_daily_metrics_first_20' AS section,
            ROW_NUMBER() OVER (ORDER BY activity_date ASC) AS position,
            'agg_daily_metrics_row' AS metric_name,
            NULL::VARCHAR AS metric_value,
            activity_date,
            dau,
            wau,
            mau,
            dau_mau_stickiness,
            total_observations,
            active_guild_members,
            guild_member_share,
            avg_observations_per_avatar,
            avg_level,
            avg_level_gain_day,
            total_level_gain_day,
            new_avatars,
            returning_avatars,
            reactivated_avatars_30d
        FROM agg_daily_metrics
        ORDER BY activity_date ASC
        LIMIT 20
    )
    SELECT * FROM summary_rows
    UNION ALL
    SELECT * FROM top_days
    UNION ALL
    SELECT * FROM first_twenty
    '''
)

con.execute(
    f"""
    COPY (
        SELECT *
        FROM daily_metrics_validation_summary
        ORDER BY
            CASE section
                WHEN 'summary' THEN 1
                WHEN 'top_10_days_by_dau' THEN 2
                WHEN 'agg_daily_metrics_first_20' THEN 3
                ELSE 4
            END,
            position
    ) TO '{validation_output_tmp}' (FORMAT CSV, HEADER)
    """
)

fct_output_tmp.replace(fct_output_path)
agg_output_tmp.replace(agg_output_path)
validation_output_tmp.replace(validation_output_path)

print('Outputs written.')



In [ ]:
final_report = con.execute(
    '''
    SELECT
        'stg_source_glob' AS metric_name,
        ? AS metric_value
    UNION ALL
    SELECT 'preview_avoided', CASE WHEN ? THEN 'yes' ELSE 'no' END
    UNION ALL
    SELECT 'stg_total_rows', CAST((SELECT COUNT(*) FROM stg_wowah_events) AS VARCHAR)
    UNION ALL
    SELECT 'stg_unique_avatar_id', CAST((SELECT COUNT(DISTINCT avatar_id) FROM stg_wowah_events) AS VARCHAR)
    UNION ALL
    SELECT 'stg_distinct_activity_dates', CAST((SELECT COUNT(DISTINCT CAST(observed_at AS DATE)) FROM stg_wowah_events) AS VARCHAR)
    UNION ALL
    SELECT 'stg_date_range', CAST((SELECT MIN(observed_at) FROM stg_wowah_events) AS VARCHAR) || ' -> ' || CAST((SELECT MAX(observed_at) FROM stg_wowah_events) AS VARCHAR)
    UNION ALL
    SELECT 'fct_player_daily_row_count', CAST((SELECT COUNT(*) FROM fct_player_daily) AS VARCHAR)
    UNION ALL
    SELECT 'agg_daily_metrics_row_count', CAST((SELECT COUNT(*) FROM agg_daily_metrics) AS VARCHAR)
    UNION ALL
    SELECT 'avg_dau', CAST((SELECT AVG(dau * 1.0) FROM agg_daily_metrics) AS VARCHAR)
    UNION ALL
    SELECT 'avg_mau', CAST((SELECT AVG(mau * 1.0) FROM agg_daily_metrics) AS VARCHAR)
    UNION ALL
    SELECT 'days_where_dau_gt_mau', CAST((SELECT SUM(CASE WHEN dau > mau THEN 1 ELSE 0 END) FROM agg_daily_metrics) AS VARCHAR)
    UNION ALL
    SELECT 'negative_level_gain_day_count', CAST((SELECT SUM(CASE WHEN level_gain_day < 0 THEN 1 ELSE 0 END) FROM fct_player_daily) AS VARCHAR)
    UNION ALL
    SELECT 'warnings', CASE
        WHEN (SELECT SUM(CASE WHEN level_gain_day < 0 THEN 1 ELSE 0 END) FROM fct_player_daily) > 0
            THEN 'negative_level_gain_day_present'
        ELSE 'none'
    END
    '''
    , [stg_source_glob, preview_avoided]
).df()

final_report
